In [49]:
# Importing libraries
# Data handling
import pandas as pd
import numpy as np

# Text processing
import re
import string

# NLP
from sklearn.feature_extraction.text import TfidfVectorizer

# Model building
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from scipy.sparse import csr_matrix
# Evaluation
from sklearn.metrics import classification_report, accuracy_score

# Saving model
import pickle




# **📥1. DATA LOADING AND EXPLORATION**
In this step, I load the dataset containing resumes and their corresponding job categories.

In [34]:
df = pd.read_csv('Data/Resume_Data.csv')
df.head()

,ID,Category,Feature
0,16852973,HR,hr administrator marketing associate hr admini...
1,22323967,HR,hr specialist hr operations summary media prof...
2,33176873,HR,hr director summary years experience recruitin...
3,27018550,HR,hr specialist summary dedicated driven dynamic...
4,17812897,HR,hr manager skill highlights hr skills hr depar...


In [50]:
df['Category'].value_counts()

Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
FINANCE                   118
ADVOCATE                  118
ACCOUNTANT                118
ENGINEERING               118
CHEF                      118
AVIATION                  117
FITNESS                   117
SALES                     116
BANKING                   115
HEALTHCARE                115
CONSULTANT                115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22
Name: count, dtype: int64

In [35]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2484 entries, 0 to 2483
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   ID        2484 non-null   int64 
 1   Category  2484 non-null   object
 2   Feature   2483 non-null   object
dtypes: int64(1), object(2)
memory usage: 58.3+ KB


In [36]:
# Checking for missing vales 
df.isnull().sum()

ID          0
Category    0
Feature     1
dtype: int64

In [38]:
df.columns

Index(['ID', 'Category', 'Feature'], dtype='object')

# **🧹 2. DATA CLEANING**
I cleaned the resume text by:
- Removing punctuation
- Removing numbers
- Converting text to lowercase
- Removing extra spaces

In [39]:
#Cleaning text data

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\d+', '', text)  # remove numbers
    text = text.translate(str.maketrans('', '', string.punctuation))  # remove punctuation
    text = re.sub(r'\s+', ' ', text)  # remove extra spaces
    return text

df['clean_resume'] = df['Feature'].apply(clean_text)

# **🧾 3. FEATURE ENGINEERING**

I convert text data into numerical format using TF-IDF (Term Frequency - Inverse Document Fre

In [40]:
# Encodeing labels
le = LabelEncoder()
df['label'] = le.fit_transform(df['Category'])

In [41]:
# TF-IDF vecorization
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)

X = tfidf.fit_transform(df['clean_resume'])
y = df['label']

# **🤖 3. MODEL TRAINING**


We train a Logistic Regression model using the processed resume text to predict job categories.

In [42]:
# Train-Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [46]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)


LogisticRegression(max_iter=1000)

# **📈 4. MODEL EVALUATION**

I evaluated the model using:
- Accuracy Score
- Precision, Recall, F1-score
- Classification Report

In [47]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=le.classes_))

Accuracy: 0.647887323943662
                        precision    recall  f1-score   support

            ACCOUNTANT       0.83      0.86      0.85        29
              ADVOCATE       0.56      0.63      0.59        30
           AGRICULTURE       1.00      0.12      0.22         8
               APPAREL       0.56      0.45      0.50        20
                  ARTS       0.15      0.17      0.16        18
            AUTOMOBILE       1.00      0.17      0.29         6
              AVIATION       0.78      0.86      0.82        21
               BANKING       0.73      0.70      0.71        23
                   BPO       0.00      0.00      0.00         2
  BUSINESS-DEVELOPMENT       0.89      0.59      0.71        27
                  CHEF       0.85      0.71      0.77        24
          CONSTRUCTION       0.90      0.76      0.83        34
            CONSULTANT       0.45      0.25      0.32        20
              DESIGNER       0.71      0.79      0.75        19
         DI

c:\Users\User\anaconda3\envs\learn-env\lib\site-packages\sklearn\metrics\_classification.py:1221: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


# **💾 5. SAVING THE MODEL**

I saved the trained model and vectorizer using pickle for deployment in a web application.

In [48]:
pickle.dump(model, open("model.pkl", "wb"))
pickle.dump(tfidf, open("tfidf.pkl", "wb"))
pickle.dump(le, open("label_encoder.pkl", "wb"))